# Gesture Detection & Clustering Pipeline - Google Colab

This notebook runs the complete gesture processing pipeline in Google Colab with GCS integration.

**Workflow:**
1. 📦 Setup environment and clone repository
2. 🔐 Authenticate with Google Drive and GCS
3. 📤 Transfer videos from Google Drive to GCS bucket
4. 🎯 Run gesture detection and clustering
5. 🤖 Train classifier on clustered data
6. 🎬 Extract animations for web app
7. 📥 Download or access results

**Prerequisites:**
- Google Cloud project with GCS enabled
- GCS bucket created (e.g., `gs://my-gesture-bucket`)
- Training videos uploaded to Google Drive

## 1. Setup Environment

Clone the repository and install dependencies.

In [ ]:
# Clone repository
!git clone https://github.com/nicoptere/pose-trainer.git
%cd pose-trainer

# Install dependencies
!pip install -q -r requirements.txt

print("✅ Environment setup complete!")

## 2. Authentication

Authenticate with Google Drive and Google Cloud Storage.

In [ ]:
from google.colab import auth, drive
import os

# Authenticate with Google Cloud
print("🔐 Authenticating with Google Cloud...")
auth.authenticate_user()

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Set up GCS credentials for gsutil
!gcloud config set project YOUR_PROJECT_ID  # Replace with your project ID

print("✅ Authentication complete!")

## 3. Configuration

Set your GCS bucket, Google Drive paths, and gesture processing parameters.

In [ ]:
import json

# GCS and Drive Configuration
GCS_BUCKET = "gs://my-gesture-bucket"  # Replace with your bucket name
DRIVE_VIDEO_PATH = "/content/drive/MyDrive/gesture_videos"  # Path to videos in Drive
GCS_VIDEO_PATH = f"{GCS_BUCKET}/videos"
GCS_OUTPUT_PATH = f"{GCS_BUCKET}/output"
GCS_MODELS_PATH = f"{GCS_BUCKET}/models"
GCS_ANIMATIONS_PATH = f"{GCS_BUCKET}/public"

# Gesture Processing Configuration
gesture_config = {
    "analysis_fps": 12,           # Analysis frame rate (lower = faster, less accurate)
    "gesture_min_seconds": 2,     # Minimum gesture duration in seconds
    "gesture_max_seconds": 6,     # Maximum gesture duration in seconds
    "n_clusters": None,           # Fixed cluster count (None = auto with HDBSCAN)
    "use_hdbscan": True,          # Auto-detect cluster count
    "dtw_downsample_factor": 1   # DTW downsampling (1 = no downsampling)
}

# Write configuration to file
with open('gesture_config.json', 'w') as f:
    json.dump(gesture_config, f, indent=2)

print(f"📊 GCS Configuration:")
print(f"  GCS Bucket: {GCS_BUCKET}")
print(f"  Drive Videos: {DRIVE_VIDEO_PATH}")
print(f"  GCS Videos: {GCS_VIDEO_PATH}")
print(f"  GCS Output: {GCS_OUTPUT_PATH}")

print(f"
⚙️  Gesture Processing:")
print(f"  Analysis FPS: {gesture_config['analysis_fps']}")
print(f"  Gesture Duration: {gesture_config['gesture_min_seconds']}-{gesture_config['gesture_max_seconds']} seconds")
print(f"  Min Frames: {gesture_config['gesture_min_seconds'] * gesture_config['analysis_fps']}")
print(f"  Max Frames: {gesture_config['gesture_max_seconds'] * gesture_config['analysis_fps']}")
print(f"  Clustering: {'HDBSCAN (auto)' if gesture_config['use_hdbscan'] else f'K-means ({gesture_config["n_clusters"]})'}")

print("
✅ Configuration saved to gesture_config.json")

## 4. Transfer Videos from Google Drive to GCS

Copy your training videos from Google Drive to GCS bucket for processing.

In [ ]:
import os
from pathlib import Path

# Check if videos exist in Drive
if not os.path.exists(DRIVE_VIDEO_PATH):
    print(f"❌ Error: Video path not found: {DRIVE_VIDEO_PATH}")
    print(f"Please upload videos to Google Drive at this location.")
else:
    # Count videos
    video_extensions = ['.mp4', '.avi', '.mov', '.MOV']
    videos = []
    for ext in video_extensions:
        videos.extend(Path(DRIVE_VIDEO_PATH).glob(f'*{ext}'))
    
    print(f"📹 Found {len(videos)} videos in Google Drive")
    
    if len(videos) > 0:
        # Upload to GCS using gsutil
        print(f"⬆️  Uploading videos to GCS: {GCS_VIDEO_PATH}")
        !gsutil -m cp -r {DRIVE_VIDEO_PATH}/* {GCS_VIDEO_PATH}/
        
        # Verify upload
        print(f"\n✅ Upload complete! Verifying...")
        !gsutil ls {GCS_VIDEO_PATH}/ | head -n 10
    else:
        print(f"❌ No videos found. Please add videos to {DRIVE_VIDEO_PATH}")

## 5. Run Gesture Detection & Clustering

Process videos to detect and cluster gestures using DTW similarity.

**This step may take 30-60 minutes depending on video count and length.**

In [ ]:
# Run gesture clustering with GCS support
print("🚀 Starting gesture detection and clustering...")
print(f"   Input: {GCS_VIDEO_PATH}")
print(f"   Output: {GCS_OUTPUT_PATH}")
print("\n⏱️  This may take 30-60 minutes for ~50 videos...\n")

!python run_gcs.py \
    --videos {GCS_VIDEO_PATH} \
    --output {GCS_OUTPUT_PATH}

print("\n✅ Clustering complete! Results saved to GCS.")

## 6. Inspect Clustering Results

Download and view the clustering manifest to see detected gesture clusters.

In [ ]:
import json

# Download manifest
manifest_gcs_path = f"{GCS_OUTPUT_PATH}/clustering_manifest.json"
!gsutil cp {manifest_gcs_path} ./clustering_manifest.json

# Load and display summary
with open('clustering_manifest.json', 'r') as f:
    manifest = json.load(f)

print("📊 Clustering Results:")
print(f"  Total gestures detected: {manifest.get('total_gestures', 0)}")
print(f"  Number of clusters: {manifest.get('num_clusters', 0)}")
print(f"  Analysis FPS: {manifest.get('analysis_fps', 0)}")

# Show cluster sizes
print("\n📦 Cluster Sizes:")
clusters = manifest.get('clusters', {})
for cluster_id, cluster_data in sorted(clusters.items()):
    gesture_count = len(cluster_data.get('gestures', []))
    print(f"  Cluster {cluster_id}: {gesture_count} gestures")

## 7. Extract Video Segments by Cluster (Optional)

Extract individual video clips for each cluster for manual review.

In [ ]:
# Extract cluster videos
print("🎬 Extracting video segments by cluster...")

!python run_gcs.py \
    --extract-clusters \
    --manifest {manifest_gcs_path}

print("\n✅ Video segments extracted to GCS!")
print(f"   Location: {GCS_OUTPUT_PATH}/cluster_*/")

## 8. Train Gesture Classifier

Train a PyTorch classifier on the clustered gestures and export to ONNX format.

In [ ]:
# Train classifier
model_output_path = f"{GCS_MODELS_PATH}/gesture_classifier.onnx"

print("🤖 Training gesture classifier...")
print(f"   Manifest: {manifest_gcs_path}")
print(f"   Output: {model_output_path}")
print("\n⏱️  This may take 5-10 minutes...\n")

!python classification_gcs.py \
    --manifest {manifest_gcs_path} \
    --output {model_output_path}

print("\n✅ Training complete! Model saved to GCS.")

## 9. Extract Cluster Animations

Extract representative gesture sequences for web app visualization.

In [ ]:
# Extract animations
animations_output_path = f"{GCS_ANIMATIONS_PATH}/cluster_animations.json"

print("🎬 Extracting cluster animations...")
print(f"   Manifest: {manifest_gcs_path}")
print(f"   Output: {animations_output_path}")

!python classification_gcs.py \
    --extract-animations \
    --manifest {manifest_gcs_path} \
    --animations-output {animations_output_path}

print("\n✅ Animations extracted and saved to GCS!")

## 10. Download Results (Optional)

Download key results to Colab or Google Drive for local inspection.

In [ ]:
# Create results directory
!mkdir -p results

# Download key files
print("📥 Downloading results...")

# Clustering manifest
!gsutil cp {manifest_gcs_path} results/

# Trained model
!gsutil cp {model_output_path} results/

# Animations
!gsutil cp {animations_output_path} results/

# Similarity report
!gsutil cp {GCS_OUTPUT_PATH}/similarity_report.md results/ 2>/dev/null || echo "No similarity report found"

print("\n✅ Results downloaded to ./results/")
!ls -lh results/

## 11. Copy Results to Google Drive (Optional)

Save results to Google Drive for persistent storage.

In [ ]:
# Copy results to Drive
drive_results_path = "/content/drive/MyDrive/gesture_results"

print(f"📁 Copying results to Google Drive: {drive_results_path}")
!mkdir -p {drive_results_path}
!cp -r results/* {drive_results_path}/

print("\n✅ Results saved to Google Drive!")

## 12. Summary & Next Steps

View the complete results summary.

In [ ]:
print("="*70)
print("🎉 GESTURE PROCESSING PIPELINE COMPLETE!")
print("="*70)

print("\n📊 Results Location:")
print(f"  GCS Bucket: {GCS_BUCKET}")
print(f"  Clustering Results: {GCS_OUTPUT_PATH}/")
print(f"  Trained Model: {model_output_path}")
print(f"  Animations: {animations_output_path}")

if os.path.exists('results/clustering_manifest.json'):
    with open('results/clustering_manifest.json', 'r') as f:
        manifest = json.load(f)
    
    print("\n📈 Statistics:")
    print(f"  Total gestures: {manifest.get('total_gestures', 0)}")
    print(f"  Clusters found: {manifest.get('num_clusters', 0)}")

print("\n🚀 Next Steps:")
print("  1. Review cluster videos in GCS")
print("  2. Integrate ONNX model into your web app")
print("  3. Use cluster_animations.json for UI visualization")
print("  4. Fine-tune clustering parameters if needed")

print("\n💡 Access your results:")
print(f"  gsutil ls {GCS_OUTPUT_PATH}/")
print(f"  gsutil ls {GCS_MODELS_PATH}/")
print(f"  gsutil ls {GCS_ANIMATIONS_PATH}/")